<a href="https://colab.research.google.com/github/VivekOPest/gaia-symbolic-regression/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 – Data Preparation and Feature Engineering

This notebook prepares the Gaia DR3 dataset for symbolic regression.

Objectives

- Load the Gaia DR3 dataset.
- Convert astronomical quantities into physical units.
- Rename variables using physically meaningful names.
- Prepare the final machine learning dataset.
- Export the processed dataset.

In [21]:
import pandas as pd
import numpy as np

from pathlib import Path

In [22]:
DATA_PATH = "gaia_sample.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

(20000, 13)


,source_id,parallax,parallax_error,parallax_over_error,ruwe,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp,teff_gspphot,radius_gspphot
0,137341754068386432,0.933154,0.051944,17.964705,1.036885,6783.598485,3.588084,16.108717,16.661566,15.408920,1.252646,4896.8936,0.8375
1,137342166385242112,4.299332,0.022674,189.618150,1.026791,38620.291417,12.237451,14.220328,15.037620,13.338170,1.699450,4397.8647,0.7090
2,137348900893898240,0.796858,0.042952,18.552425,1.002285,9817.506707,3.566114,15.707364,16.114141,15.123431,0.990710,5307.8760,0.7948
3,137351615313312384,1.354746,0.018821,71.981310,1.202068,262440.355661,55.139145,12.139791,12.538257,11.568266,0.969991,5952.5596,2.3795
4,137355326165033472,1.222841,0.056203,21.757740,1.006233,4903.220496,2.882057,16.461163,17.241302,15.604395,1.636908,4525.4976,0.8585


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   source_id               20000 non-null  int64  
 1   parallax                20000 non-null  float64
 2   parallax_error          20000 non-null  float64
 3   parallax_over_error     20000 non-null  float64
 4   ruwe                    20000 non-null  float64
 5   phot_g_mean_flux        20000 non-null  float64
 6   phot_g_mean_flux_error  20000 non-null  float64
 7   phot_g_mean_mag         20000 non-null  float64
 8   phot_bp_mean_mag        20000 non-null  float64
 9   phot_rp_mean_mag        20000 non-null  float64
 10  bp_rp                   20000 non-null  float64
 11  teff_gspphot            20000 non-null  float64
 12  radius_gspphot          20000 non-null  float64
dtypes: float64(12), int64(1)
memory usage: 2.0 MB


## Computing Physical Quantities

The Gaia catalogue provides stellar parallax in milliarcseconds (mas) and stellar radius in units of the solar radius.

The following physical quantities are computed:

Distance

\[
d=\frac{1000}{p}
\]

where

- \(d\) is the distance in parsecs,
- \(p\) is the parallax in milliarcseconds.

The distance is subsequently converted to metres using

\[
1\ \mathrm{pc}=3.085677581\times10^{16}\ \mathrm{m}.
\]

Similarly,

\[
1\ R_\odot = 6.957\times10^8\ \mathrm{m}.
\]

Temperature is already provided in Kelvin.

The Gaia G-band flux is retained as an observational proxy for stellar flux.

In [24]:
PARSEC_TO_M = 3.085677581e16

SOLAR_RADIUS_TO_M = 6.957e8

df["distance_m"] = (
    1000 / df["parallax"]
) * PARSEC_TO_M

df["radius_m"] = (
    df["radius_gspphot"]
    * SOLAR_RADIUS_TO_M
)

In [26]:
processed = pd.DataFrame({

    "temperature_K": df["teff_gspphot"],

    "distance_m": df["distance_m"],

    "radius_m": df["radius_m"],

    "flux_proxy": df["phot_g_mean_flux"]

})

In [27]:
processed.info()

processed.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   temperature_K  20000 non-null  float64
 1   distance_m     20000 non-null  float64
 2   radius_m       20000 non-null  float64
 3   flux_proxy     20000 non-null  float64
dtypes: float64(4)
memory usage: 625.1 KB


,0
temperature_K,0
distance_m,0
radius_m,0
flux_proxy,0


In [28]:
processed = processed.dropna()

print(processed.shape)

(20000, 4)


In [29]:
processed.head()

,temperature_K,distance_m,radius_m,flux_proxy
0,4896.8936,3.306717e+19,5.826488e+08,6783.598485
1,4397.8647,7.177110e+18,4.932513e+08,38620.291417
2,5307.8760,3.872305e+19,5.529424e+08,9817.506707
3,5952.5596,2.277680e+19,1.655418e+09,262440.355661
4,4525.4976,2.523367e+19,5.972584e+08,4903.220496


In [30]:
processed.to_csv(
    "gaia_features.csv",
    index=False
)

In [31]:
from google.colab import files

files.download("gaia_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>